# Digital Twin
This project is made to take your place and answer on your behalf when it comes to your LinkedIn profile.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import json
import os
from pathlib import Path
import gradio as gr

In [155]:
load_dotenv(override=True)

True

In [ ]:
groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"Groq API Key exists and begins with {groq_api_key[:6]}\n")
else:
    print(f"Groq API Key not set.\n")

In [ ]:
openrouter_api = os.getenv('OPENROUTER_API_KEY')
if openrouter_api:
    print(f"OpenRouter API key exists and begins with {openrouter_api[:6]}\n")
else:
    print(f"OpenRouter API key not set.\n")

In [ ]:
resend_api_key = os.getenv('resend.api_key')
if groq_api_key:
    print(f"Resend API Key exists and begins with {resend_api_key[:6]}\n")
else:
    print(f"Resend API Key not set.\n")

In [ ]:
pdf_file = Path("linkedin.pdf")
summary_file = Path("summary.txt")

In [154]:
reader = PdfReader(pdf_file)

In [152]:
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [33]:
with open(summary_file, "r", encoding= "utf-8") as f:
    summary = f.read()

In [108]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
display(Markdown(system_prompt))

In [110]:
messages = [
    {"role":"system", "content":system_prompt},
    {"role":"system", "content":"Hi! Please tell me about yourself."}
]

# Models

We'll try a few different models so we can compare how they sound as the digital twin:

- **GPT-OSS 20B** via Groq
- **Nemotron 3 Ultra** and **Laguna S 2.1** via OpenRouter
- **Llama 3.2** running locally with Ollama

Give each one a name below so we can pick them easily later.

In [38]:
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
groq = OpenAI(
    api_key= groq_api_key,
    base_url= GROQ_BASE_URL
)

In [40]:
openrouter = OpenAI(
    api_key= openrouter_api,
    base_url= OPENROUTER_BASE_URL
)

In [42]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest ⠋ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕█

In [43]:
ollama = OpenAI(
    api_key= "anything",
    base_url= OLLAMA_BASE_URL
)

In [ ]:
groq_model_name = "openai/gpt-oss-20b"
openrouter_nvidia_model_name = "nvidia/nemotron-3-ultra-550b-a55b:free"
openrouter_laguna_model_name = "poolside/laguna-s-2.1:free"
ollama_model_name = "llama3.2"

In [ ]:
ollama_response = ollama.chat.completions.create(model= ollama_model_name, messages= messages)
ollama_answer = ollama_response.choices[0].message.content

display(Markdown(ollama_answer))

In [ ]:
groq_response = groq.chat.completions.create(model= groq_model_name, messages= messages)
groq_answer = groq_response.choices[0].message.content

display(Markdown(groq_answer))

In [ ]:
nvidia_response = openrouter.chat.completions.create(model= openrouter_nvidia_model_name, messages= messages)
nvidia_answer = nvidia_response.choices[0].message.content

display(Markdown(nvidia_answer))

In [ ]:
laguna_response = openrouter.chat.completions.create(model= openrouter_laguna_model_name, messages= messages)
laguna_answer = laguna_response.choices[0].message.content

display(Markdown(laguna_answer))

# Email recorder tool

In [141]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [142]:
record_email_tool("test@gmail.com")

Tool called to record an email: test@gmail.com


'Email received'

# JSON to describe the tool to the bot

In [143]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [144]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [145]:
tools

[{'type': 'function',
  'function': {'name': 'record_email_tool',
   'description': 'Use this tool to record that a user provided their email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'}},
    'required': ['email'],
    'additionalProperties': False}}}]

# A chat() function with the addition of the tools

# Multiple email appendance chance

In [149]:
def chat(message, history, model):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    if model == groq_model_name:
        client = groq
    elif model == openrouter_nvidia_model_name:
        client = openrouter
    elif model == openrouter_laguna_model_name:
        client = openrouter
    else:
        client = ollama

    response = client.chat.completions.create(model=model, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)

        for tool_call in message.tool_calls:
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})

        response = client.chat.completions.create(model=model, messages=messages, tools=tools)

    return response.choices[0].message.content

In [150]:
model_dropdown = gr.Dropdown(
    choices=[
        ("GPT-OSS 20B", groq_model_name),
        ("NVIDIA Nemotron 3 Ultra", openrouter_nvidia_model_name),
        ("Laguna S 2.1", openrouter_laguna_model_name),
        ("Llama 3.2", ollama_model_name),
    ],
    value=groq_model_name,
    label="Choose model",
)

In [ ]:
gr.ChatInterface(
    chat,
    additional_inputs=[model_dropdown],
).launch(inbrowser=False)